# Block 1 · EDA & the anatomy of missing data
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA / SMMD-AAB)
> Course anchor dataset: retail-credit / PD portfolio

**Business framing.** We are a lender. Each row is a credit customer; the target `default` = 1 means the customer defaulted. Before we model anything, we practise the first two phases of **CRISP-DM**: *business understanding* and *data understanding*. Good data mining lives or dies here.

**Learning goals for this lab**
1. Make *first contact* with a dataset: shape, dtypes, keys, target balance.
2. Read univariate and bivariate structure — and spot implausible values.
3. Map correlations and outliers without fooling yourself.
4. Diagnose **missingness** and classify it (MCAR / MAR / MNAR / structural).
5. Run the imputation experiment: see *why* naive imputation is dangerous in a regulated credit setting.

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_credit
# source="synthetic": deterministic and offline-safe; the column names below
# match the synthetic schema. (source="auto" may fetch German Credit instead,
# which has a different schema.)
df = load_credit(source="synthetic")
df.shape

## 1. First contact
Four commands before anything else — the same four from the lecture. You are looking for: numbers stored as strings, phantom category levels, minima/maxima that make no physical sense, and `describe()` counts smaller than `len(df)` (that is missingness).

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T.round(2)

### 1a. Keys and duplicates
**What should uniquely identify a row?** Here: `customer_id` — one row = one customer with one loan application (the *unit of analysis*). Verify it, don't assume it: duplicated rows are double-counted evidence, and a duplicate that lands in both train and test set inflates every score.

In [ ]:
print("customer_id unique?", df['customer_id'].is_unique)
print("full-row duplicates:", df.duplicated().sum())

## 2. The target comes first
Class imbalance is the norm in credit — note it now, it will shape every evaluation choice later. A model that always predicts "repaid" is right about 7 times out of 10 here, and completely useless. (Real PD portfolios are far more extreme: 1–5% default rates.)

In [ ]:
target_rate = df['default'].mean()
print(f'Default rate: {target_rate:.1%}')
df['default'].value_counts().plot(kind='bar', edgecolor='white',
                                  title='Target balance (0=repaid, 1=default)')
plt.show()

> **Leakage warning.** The column `collections_contact` is in this dataset on purpose. It is only known *after* an account goes bad, so using it as a predictor would leak the answer. Apply the **time-travel test** to every feature: *"would I have known this value at the moment of the decision?"* We keep the column to demonstrate leakage in Block 3 — **do not use it as a feature.**

## 3. Univariate structure (numeric)
`describe()` is your friend. Look for implausible values, heavy tails, and units.

In [ ]:
num_cols = ['age', 'monthly_income_pln', 'employment_length_years',
            'loan_amount_pln', 'credit_utilization', 'debt_to_income',
            'savings_balance_pln', 'num_existing_loans']
df[num_cols].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(),
                   ['age', 'monthly_income_pln', 'debt_to_income',
                    'credit_utilization', 'loan_amount_pln', 'employment_length_years']):
    df[col].plot(kind='hist', bins=30, ax=ax, edgecolor='white')
    ax.set_title(col)
fig.tight_layout(); plt.show()

### 3a. Skew: mean vs median
Income is right-skewed: a few high earners pull the mean above the median. Skew tells you which summary to report — and whether a transform will help.

In [ ]:
inc = df['monthly_income_pln'].dropna()
print(f"mean   = {inc.mean():>10,.0f} PLN")
print(f"median = {inc.median():>10,.0f} PLN")
print(f"skewness = {inc.skew():.2f}   (0 = symmetric)")

### 3b. Heavy tails? Change the lens
Monetary amounts almost always deserve a log look — money behaves *multiplicatively*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(inc, bins=50, edgecolor='white')
axes[0].set_title('raw scale')
axes[1].hist(np.log10(inc), bins=50, edgecolor='white')
axes[1].set_title('log10 scale')
fig.tight_layout(); plt.show()

### 3c. Sanity checks: implausible values
Every red flag below is a *question for the data owner*, not a value to silently "fix". None of these should trigger in our portfolio — but the checks cost three lines and have saved careers.

In [ ]:
checks = {
    "age outside [18, 100]":        ((df['age'] < 18) | (df['age'] > 100)).sum(),
    "negative income":              (df['monthly_income_pln'] < 0).sum(),
    "sentinel income (>= 999999)":  (df['monthly_income_pln'] >= 999_999).sum(),
    "utilisation > 1.5":            (df['credit_utilization'] > 1.5).sum(),
    "constant columns":             int((df.nunique(dropna=False) <= 1).sum()),
}
pd.Series(checks, name="violations")

### 3d. Categoricals: levels and rare categories
Check the levels: typos and case variants create phantom categories. Rare levels (< 1–2% of rows) are often grouped into `other` — but only *after* checking they are not special (e.g. a tiny, very risky segment).

In [ ]:
for col in ['housing', 'purpose', 'region', 'checking_status']:
    print(f"--- {col}")
    print(df[col].value_counts(normalize=True).round(3).to_string(), "\n")

## 4. Bivariate structure — which features move with default?
A quick, model-free signal check: average default rate across bins / categories. Compare each group to the **portfolio average**, and mind small groups — a rate computed on 30 customers is noise.

In [ ]:
# categorical driver
ax = df.groupby('checking_status')['default'].mean().sort_values().plot(
    kind='barh', title='Default rate by checking-account status')
ax.axvline(df['default'].mean(), linestyle='--', color='gray')
plt.xlabel('default rate'); plt.show()

In [ ]:
# numeric driver, binned
df['dti_bin'] = pd.qcut(df['debt_to_income'], 5, duplicates='drop')
df.groupby('dti_bin', observed=True)['default'].mean().plot(
    kind='bar', title='Default rate by debt-to-income quintile')
plt.ylabel('default rate'); plt.tight_layout(); plt.show()

### 4a. A first map: correlations
One picture of which features move together — and which move with the target. Three caveats from the lecture: correlation is not causation; Pearson sees only straight lines; one outlier can manufacture correlation. A correlation matrix generates **hypotheses**, not conclusions.

In [ ]:
corr_cols = num_cols + ['default']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)));  ax.set_xticklabels(corr_cols, rotation=45, ha='right')
ax.set_yticks(range(len(corr_cols)));  ax.set_yticklabels(corr_cols)
fig.colorbar(im, label='Pearson correlation')
plt.tight_layout(); plt.show()

corr['default'].drop('default').sort_values(key=abs, ascending=False).round(3)

### 4b. Outliers: spot, then decide
The boxplot's **IQR rule** flags points beyond `Q3 + 1.5·IQR`. It is a convention, not a law — for skewed variables it flags many perfectly genuine values. The decision tree from the lecture: investigate → error? fix or set missing → genuine? keep (maybe cap / log) → document the rule. And remember the fraud-detection perspective: sometimes the outliers *are* the signal.

In [ ]:
q1, q3 = inc.quantile([0.25, 0.75])
iqr = q3 - q1
fence = q3 + 1.5 * iqr
flagged = (inc > fence).sum()
print(f"upper fence = {fence:,.0f} PLN;  flagged: {flagged} customers "
      f"({flagged/len(inc):.1%}) — all genuine high earners here")

plt.figure(figsize=(9, 2.5))
plt.boxplot(inc, orientation='horizontal', widths=0.5)
plt.axvline(fence, linestyle=':', color='red')
plt.xlabel('monthly income (PLN)'); plt.yticks([])
plt.tight_layout(); plt.show()

## 5. Missingness — the part everyone skips
First, *how much* is missing and *where*?

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].round(3)

### 5a. Seeing the pattern
A missingness matrix shows *structure*: which columns, how often, and whether gaps co-occur in the same rows. No extra library needed — `df.isna()` is the whole trick.

In [ ]:
sample = df.sample(300, random_state=1).reset_index(drop=True)
plt.figure(figsize=(10, 4.5))
plt.imshow(sample.notna().T, aspect='auto', cmap='Blues', vmin=-0.4, vmax=1.4,
           interpolation='nearest')
plt.yticks(range(df.shape[1]), df.columns, fontsize=8)
plt.xlabel('customers (sample of 300)')
plt.title('white = missing value')
plt.tight_layout(); plt.show()

### 5b. A taxonomy of missingness
Missing values are **not** all the same. The mechanism decides whether it is safe to drop, impute, or *model* them:

| Type | Meaning | Example in this portfolio |
|---|---|---|
| **MCAR** | Missing Completely At Random — unrelated to anything | a form field lost to a system glitch |
| **MAR** | Missing At Random — depends on *observed* data | `employment_length` missing more for renters |
| **MNAR** | Missing Not At Random — depends on the *unobserved* value itself | high earners decline to state `income` |
| **Structural** | The value cannot exist | `months_since_last_delinquency` for customers who were never delinquent |

Let's find evidence for each.

In [ ]:
# MAR check: is employment_length missingness related to housing?
df['emp_missing'] = df['employment_length_years'].isna()
df.groupby('housing')['emp_missing'].mean().round(3)

In [ ]:
# MNAR is invisible *in principle*: the missingness depends on the value we
# never observe. But it often leaves fingerprints on correlated, observed
# variables. Savings correlate with income — so if income is missing mostly
# for high earners, the income-missing group should show HIGHER savings:
df['inc_missing'] = df['monthly_income_pln'].isna()
df.groupby('inc_missing')[['savings_balance_pln', 'loan_amount_pln']].median().round(0)

The income-missing group has visibly higher savings — indirect evidence that the *unstated incomes were high*, i.e. MNAR. In real projects this diagnosis comes from proxies like this plus domain knowledge (here: wealthy applicants often decline to state income); no statistical test can prove MNAR from the observed data alone.

In [ ]:
# Structural: months_since_last_delinquency is NaN exactly when there was no delinquency.
structural = df['months_since_last_delinquency'].isna()
print(f'{structural.mean():.1%} of rows have NO prior delinquency (structural NaN).')
# The *fact* of missingness is itself predictive — compare default rates:
print(df.groupby(structural)['default'].mean().round(3))

### 5c. Deletion — when dropping is (not) safe
Dropping rows (listwise deletion) is safe-ish only under **MCAR**: you lose power, but no bias. Under MAR/MNAR it biases the sample. Check what dropping income-missing rows would do here:

In [ ]:
kept = df[~df['inc_missing']]
print(f"rows lost: {df['inc_missing'].mean():.1%}")
print(f"median savings, full sample: {df['savings_balance_pln'].median():,.0f}")
print(f"median savings, after drop:  {kept['savings_balance_pln'].median():,.0f}")
print("-> dropping income-missing rows removes disproportionately wealthy customers (MNAR bias).")

## 6. The experiment: mean-impute vs indicator-plus-impute
This is the punchline of Block 1, hands-on. `months_since_last_delinquency` is structurally missing for customers who were **never delinquent** — the safest group in the portfolio. Watch what mean imputation does to them.

In [ ]:
col = 'months_since_last_delinquency'
col_mean = df[col].mean()
print(f"mean of observed values: {col_mean:.1f} months")

# --- Version A: naive mean imputation -------------------------------
version_a = df[[col, 'default']].copy()
version_a['imputed'] = version_a[col].fillna(col_mean)

# After imputation, two very different groups carry the SAME feature value:
near_mean = version_a[col].sub(col_mean).abs() < 5   # truly delinquent ~mean months ago
was_nan = version_a[col].isna()                       # never delinquent at all
print(f"default rate | true value near the mean : {version_a.loc[near_mean, 'default'].mean():.1%}")
print(f"default rate | imputed (never delinquent): {version_a.loc[was_nan, 'default'].mean():.1%}")
print("-> identical feature value, very different risk: the model can no longer tell them apart.")
print("-> and we have invented a delinquency ~28 months ago for customers who never had one.")

In [ ]:
# --- Version B: missing-indicator first, then impute -----------------
df['delinq_missing'] = df[col].isna().astype(int)
df['income_missing'] = df['monthly_income_pln'].isna().astype(int)
df['delinq_imputed'] = df[col].fillna(col_mean)

# The indicator preserves exactly the signal that imputation erased:
df.groupby('delinq_missing')['default'].mean().round(3)

In [ ]:
# Model-free quantification: how much default signal does each carrier hold?
signal = {
    'mean-imputed value (version A)': version_a['imputed'].corr(version_a['default']),
    'imputed value (version B)':      df['delinq_imputed'].corr(df['default']),
    'missing-indicator (version B)':  df['delinq_missing'].corr(df['default']),
}
pd.Series(signal, name='corr with default').round(3)

### 6a. Why this matters in regulated lending
* Blindly mean-imputing `months_since_last_delinquency` *destroys* a real signal and invents a delinquency history that never happened.
* The **missing-indicator** feature is often the honest choice — and it must be explainable, because regulators expect adverse-action reasons to be truthful. "You were declined because of a delinquency 28 months ago" would be a *false statement* for a never-delinquent customer.
* Golden rule for later blocks: fit any imputation on the **training data only** — a preview of Block 3's leakage discussion.

We will return to this when we discuss GBDT handling of missing values and model explanations later in the course.

## 6b. Feature engineering: hands-on
The indicator you just built *is* feature engineering. The toolbox has five recurring moves — ratio, flag, binning, interaction, date arithmetic — all disciplined by two rules: application-time information only (the time-travel test), and transforms fitted on training data only. Let's try an **interaction**: renting alone and no-checking-account alone are each mildly risky; watch what their combination does.

In [ ]:
df['renter_no_checking'] = (df['housing'].eq('rent')
                            & df['checking_status'].eq('none')).astype(int)

segments = {
    'whole portfolio':            df['default'].mean(),
    'renter':                     df.loc[df['housing'].eq('rent'), 'default'].mean(),
    'no checking acct':           df.loc[df['checking_status'].eq('none'), 'default'].mean(),
    'renter AND no checking':     df.loc[df['renter_no_checking'] == 1, 'default'].mean(),
}
n_seg = (df['renter_no_checking'] == 1).sum()
print(f"segment size: {n_seg} customers")
pd.Series(segments, name='default rate').round(3)

Two weak signals, combined by domain sense, concentrate into a strong one — and the segment is large enough (a few hundred customers) to trust the rate.

**Encodings preview.** Most models eat numbers, not labels. The scale of measurement picks the encoding: one-hot for nominal, ordinal codes only where order is real, target encoding / WoE fitted on training data only (Block 2 takes over from here).

In [ ]:
# One-hot encoding a nominal variable — one 0/1 column per level:
pd.get_dummies(df[['housing']], prefix='housing').head()

In [ ]:
# Ordinal encoding where the order is REAL (checking status has one):
checking_order = {'none': 0, 'low': 1, 'medium': 2, 'high': 3}
df['checking_ord'] = df['checking_status'].map(checking_order)
df[['checking_status', 'checking_ord']].drop_duplicates().sort_values('checking_ord')

## 7. Exercises
1. Build a bivariate plot of default rate by `credit_utilization` **decile** (like the lecture figure). Is the relationship monotone?
2. Is `monthly_income_pln` missingness related to `region`? What mechanism would that suggest?
3. Compute the Spearman correlation of the numeric features with `default` and compare with the Pearson values from section 4a. Where do they differ most, and why might that be?
4. Apply the IQR rule to `debt_to_income`. Would you cap, keep, or investigate the flagged values? Justify in one sentence.
5. Propose one feature you would engineer from `loan_amount_pln`, `loan_term_months` and `monthly_income_pln`. Why would it help a default model?

*Hand-in for the project team: a one-paragraph 'data understanding' note describing the target, the class balance, and the top three data-quality risks you found.*

**"Done" for today** = you can defend, in one paragraph, how you would treat each missing column and why.

In [ ]:
# scratch cell for the exercises